# Deep Agents Lab — Experiment Notebook

This notebook is your mixing deck for experimenting with Deep Agents.
Change a prompt, swap a model, edit a skill — then re-run the cell and compare.

**How to use:**
1. Run Cell 1 (Setup) once
2. Pick any experiment cell below
3. Change the highlighted variable
4. Re-run just that cell
5. Compare the output

## Cell 1: Setup

Run this once to initialize. It reads your model choice from the `DEEPAGENTS_MODEL` environment variable.

In [ ]:
import os
from deepagents import create_deep_agent
from langchain_core.tools import tool

MODEL = os.environ.get("DEEPAGENTS_MODEL", "anthropic:claude-sonnet-4-6")
print(f"Using model: {MODEL}")

---
## Experiment 1: System Prompt Engineering

Change the `SYSTEM_PROMPT` below and re-run the cell to see how it affects the agent's behavior.

**Try these variations:**
- Change the persona ("security expert", "junior developer", "CTO")
- Add constraints ("respond in bullet points only", "limit to 50 words")
- Change the tone ("formal", "casual", "sarcastic")

In [ ]:
# === CHANGE THIS ===
SYSTEM_PROMPT = """You are a senior Python developer who cares deeply about
clean code and best practices. When reviewing code, be specific and constructive."""

USER_MESSAGE = """Review this function:

def process(data):
    result = []
    for i in range(len(data)):
        if data[i] != None:
            result.append(data[i].upper())
    return result
"""
# === END CHANGE ===

agent = create_deep_agent(model=MODEL, system_prompt=SYSTEM_PROMPT)
result = agent.invoke({"messages": [("user", USER_MESSAGE)]})
print(result["messages"][-1].content)

---
## Experiment 2: Custom Skills Inline

This experiment creates a skill on disk, registers it with the agent, and runs a task.
Edit the skill content and re-run to see how it changes the agent's approach.

**Try these variations:**
- Change the review focus (security-only, performance-only, readability-only)
- Add or remove checklist items
- Change the output format (table, bullet points, numbered list)

In [ ]:
import os
from pathlib import Path

# === CHANGE THIS ===
SKILL_CONTENT = """---
name: quick-review
description: Quick code review focusing on the most critical issues only.
---

# Quick Review Skill

## Focus Areas (check in this order)
1. Security vulnerabilities (injection, hardcoded secrets)
2. Bugs that would cause crashes or data loss
3. Performance issues that would affect users

## Output Format
For each issue found:
- Severity: CRITICAL / WARNING / INFO
- Line: approximate location
- Issue: one-sentence description
- Fix: one-sentence suggestion

If no issues found, say "Looks good!" and explain why.
"""
# === END CHANGE ===

# Write the skill to disk
skill_dir = Path("lab-skills/quick-review")
skill_dir.mkdir(parents=True, exist_ok=True)
(skill_dir / "SKILL.md").write_text(SKILL_CONTENT)

# Create agent with the skill
agent = create_deep_agent(model=MODEL, skills=["./lab-skills/"])
result = agent.invoke({"messages": [("user",
    "Using your quick-review skill, review this code:\n\n"
    "import subprocess\n"
    "def run_cmd(user_input):\n"
    "    return subprocess.run(user_input, shell=True, capture_output=True)\n"
)]})
print(result["messages"][-1].content)

---
## Experiment 3: Model Comparison

Run the same prompt against different models and compare quality, speed, and style.

**Try these:**
- `anthropic:claude-sonnet-4-6` (default, high quality)
- `anthropic:claude-haiku-4-5-20251001` (fast, cheaper)
- `ollama:llama3.1:8b` (local, no API cost)
- Any model on your `OPENAI_API_BASE` endpoint

In [ ]:
import time

# === CHANGE THIS ===
COMPARE_MODELS = [
    MODEL,  # your default model
    # "anthropic:claude-haiku-4-5-20251001",  # uncomment to compare
    # "ollama:llama3.1:8b",                   # uncomment for local
]

PROMPT = "Explain what a Python decorator is in exactly 3 sentences."
# === END CHANGE ===

for model_name in COMPARE_MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    
    start = time.time()
    agent = create_deep_agent(model=model_name)
    result = agent.invoke({"messages": [("user", PROMPT)]})
    elapsed = time.time() - start
    
    print(result["messages"][-1].content)
    print(f"\n⏱ {elapsed:.1f}s")

---
## Experiment 4: Subagent Routing

Change the subagent descriptions and observe how the main agent routes tasks differently.

**Try:**
- Make descriptions more/less specific
- Add a third subagent
- Change which subagent handles the test prompt

In [ ]:
# === CHANGE THESE ===
subagents = [
    {
        "name": "researcher",
        "description": "Use for factual questions that need thorough investigation.",
        "system_prompt": "You are a research assistant. Be thorough and cite sources.",
    },
    {
        "name": "creative",
        "description": "Use for creative writing, brainstorming, and generating ideas.",
        "system_prompt": "You are a creative writer. Be imaginative and original.",
    },
]

TEST_PROMPT = "Write a short explanation of how DNS works"
# === END CHANGE ===

agent = create_deep_agent(model=MODEL, subagents=subagents)
result = agent.invoke({"messages": [("user", TEST_PROMPT)]})

# Show which subagent was used
for msg in result["messages"]:
    if hasattr(msg, 'tool_calls'):
        for tc in msg.tool_calls:
            if tc['name'] == 'task':
                print(f"Delegated to: {tc['args'].get('subagent_type')}")
                print(f"Task: {tc['args'].get('description', '')[:100]}...")
                print()

print(result["messages"][-1].content)